## Column masking

A **column mask** is a function that runs at query time on a specific column and decides — typically from the current user / group — whether to return the real value or a masked one.

**Step 1 — declare the mask as a UC function:**

```sql
CREATE OR REPLACE FUNCTION fintech_dev.security.mask_pan(pan STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('fraud_analysts') THEN concat('XXXX-XXXX-XXXX-', right(pan, 4))
  WHEN is_account_group_member('compliance')     THEN pan          -- full value
  ELSE 'REDACTED'
END;
```

**Step 2 — attach it to a column:**

```sql
ALTER TABLE fintech_dev.silver.card_accounts
  ALTER COLUMN pan SET MASK fintech_dev.security.mask_pan;
```

After that, **every read** of `card_accounts.pan` runs through `mask_pan` — no matter the query path or BI tool. The function has access to `current_user()`, `is_account_group_member(...)`, and the other security functions.

**Remove it:** `ALTER TABLE ... ALTER COLUMN pan DROP MASK;`.

**The exam tell:** *"hide the PAN from analysts but show last-4 to the fraud team"* → a **column mask** that branches on `is_account_group_member`. It's applied to the **base table**, so there's no "forgot to use the secure view" escape hatch — the mask travels with the column.
